《检查 crRNA 的长度、方向、GC% 和 RNAfold MFE》

In [1]:
import RNA

In [2]:
def analyze_crrna(
    crrna_seq:str,
    scaffold_seq:str="AAUUUCACUACUAGUGAAGA",
    expected_spacer_len:int=20,
):
    """分析LbCas12a crRNA 的长度、方向、GC% 和RNAfold MFE
    Parameters:
    -----------
    crrna_seq:str
        输入的全长 crRNA 序列 (RNA,5'->3')
    scaffold_seq:str
        LbCas12a 标准 5' Scaffold 序列
    expected_spacer_len:int
        预期的 Spacer 长度(通常20nt)

    Reterns:
    --------
    dict:包含检测和分析结果
    """

转换为大写并处理为空格

In [4]:
crrna_seq = "AAUUUCACUACUAGUGAAGAUAGCACCAUCUGAAAUCGGU"
scaffold_seq = "AAUUUCACUACUAGUGAAGA"

seq = crrna_seq.upper().strip().replace("T", "U")
scaffold = scaffold_seq.upper().strip().replace("T", "U")

results = {}
print("清洗后的 crRNA 序列:", seq)

清洗后的 crRNA 序列: AAUUUCACUACUAGUGAAGAUAGCACCAUCUGAAAUCGGU


检查方向与 Scaffold 合法性 (5' -> 3')

In [6]:
if seq.startswith(scaffold):
        results["orientation_valid"] = True
        results["orientation_msg"] = (
            "方向正确 (5' 端包含正确的 LbCas12a Scaffold)"
        )

3'端的Spacer

In [8]:
# 3. 校验方向与切割 Spacer 
if seq.startswith(scaffold):
    results["orientation_valid"] = True
    results["orientation_msg"] = "方向正确 (5' 端包含正确的 LbCas12a Scaffold)"
    spacer = seq[len(scaffold) :]
else:
    results["orientation_valid"] = False
    results["orientation_msg"] = (
        "警告: 5' 端未检测到标准 Scaffold，请检查序列方向或是否反向互补！"
    )
    spacer = seq

计算基本参数

In [9]:
results["full_length"] = len(seq)
results["spacer"] = spacer
results["spacer_length"] = len(spacer)

计算 GC%

In [10]:
g_c_count = spacer.count("G") + spacer.count("C")
results["spacer_gc_percent"] = (
    round(g_c_count / len(spacer) * 100, 2) if spacer else 0.0
)

计算MFE

In [11]:
fc_full = RNA.fold_compound(seq)
struct_full, mfe_full = fc_full.mfe()

fc_spacer = RNA.fold_compound(spacer)
struct_spacer, mfe_spacer = fc_spacer.mfe()

results["full_structure"] = struct_full
results["full_mfe"] = round(mfe_full, 2)
results["spacer_structure"] = struct_spacer
results["spacer_mfe"] = round(mfe_spacer, 2)

结果是：

In [12]:
print("=== crRNA 结构与热力学检查结果 ===")
print("方向检测  :", results["orientation_msg"])
print(
    f"长度统计  : 全长 {results['full_length']} nt | Spacer {results['spacer_length']} nt"
)
print(f"Spacer GC% : {results['spacer_gc_percent']}%")
print(
    f"全长 MFE   : {results['full_mfe']} kcal/mol (二级结构: {results['full_structure']})"
)
print(
    f"Spacer MFE : {results['spacer_mfe']} kcal/mol (二级结构: {results['spacer_structure']})"
)

=== crRNA 结构与热力学检查结果 ===
方向检测  : 方向正确 (5' 端包含正确的 LbCas12a Scaffold)
长度统计  : 全长 40 nt | Spacer 20 nt
Spacer GC% : 45.0%
全长 MFE   : -5.1 kcal/mol (二级结构: ...((((((...))))))(((..((.....))..)))...)
Spacer MFE : -0.3 kcal/mol (二级结构: ....(((..........))))


最后，确认是否是由正确的 RPA 产物和 PAM 推导出来的d=====(￣▽￣*)b 

In [15]:
target_mir29a = "UAGCACCAUCUGAAAUCGGUUA"

print("目标长度:", len(target_mir29a))
print("Spacer长度:", len(seq))

目标长度: 22
Spacer长度: 40
